# JSEP Kaggle Notebook — Vehicle Detector Training (YOLOv26)
This notebook merges uploaded YOLO datasets, validates labels, and trains a vehicle detector using YOLOv26 on a single Kaggle T4 accelerator.


In [ ]:
import os
import sys
import json
import shutil
from pathlib import Path

KAGGLE_INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
DATASETS = WORK / 'datasets'
MERGED = DATASETS / 'merged_vehicles'
RUNS = WORK / 'jsep_runs'
ARTIFACTS = WORK / 'artifacts'

for p in [DATASETS, MERGED, RUNS, ARTIFACTS]:
    p.mkdir(parents=True, exist_ok=True)

ROBOFLOW_API_KEY = os.environ.get('ROBOFLOW_API_KEY', '')
print('Python:', sys.version.splitlines()[0])
print('Kaggle input exists:', KAGGLE_INPUT.exists())
print('Roboflow key set:', bool(ROBOFLOW_API_KEY))
print('Working directories:')
print(' -', DATASETS)
print(' -', MERGED)
print(' -', RUNS)
print(' -', ARTIFACTS)


## Install Dependencies
Run these in a Kaggle shell cell if needed:
````bash
pip -q install ultralytics roboflow pyyaml pymupdf tqdm
pip -q install google-generativeai chromadb
````


In [ ]:
# Discover uploaded YOLO dataset roots under /kaggle/input
uploaded_dirs = []
for path in sorted(KAGGLE_INPUT.glob('**/data.yaml')) + sorted(KAGGLE_INPUT.glob('**/dataset.yaml')):
    uploaded_dirs.append(path.parent)

if uploaded_dirs:
    print('Found YOLO dataset roots:')
    for d in uploaded_dirs:
        print('-', d)
else:
    print('No YOLO datasets found in /kaggle/input.')
    print('Upload a dataset with data.yaml/dataset.yaml, or set ROBOFLOW_API_KEY and adapt roboflow_datasets.')

# Optional: download from Roboflow if you have a key and dataset mapping.
# from roboflow import Roboflow
# rf = Roboflow(api_key=ROBOFLOW_API_KEY)
# roboflow_datasets = [
#     ('your-workspace', 'your-vehicle-dataset', 1),
# ]
# for workspace, project, version in roboflow_datasets:
#     target = DATASETS / project
#     if not target.exists():
#         rf.workspace(workspace).project(project).version(version).download('yolov8', location=str(target))
#     uploaded_dirs.append(target)

print('')
print('Use the uploaded dataset roots above or add your own vehicle dataset to /kaggle/input.')


In [ ]:
# Merge YOLO-format datasets into a single merged_vehicles dataset and validate labels
import yaml

def find_dataset_yaml(root: Path) -> Path:
    candidates = list(root.glob('**/data.yaml')) + list(root.glob('**/dataset.yaml'))
    if not candidates:
        raise FileNotFoundError(f'No data.yaml found under {root}')
    return candidates[0]

def normalize_names(names):
    if isinstance(names, list):
        return {i: v for i, v in enumerate(names)}
    return names or {}

def copy_split(src_root: Path, split: str, dst_root: Path) -> int:
    image_dirs = [src_root / split / 'images', src_root / 'images' / split, src_root / split]
    label_dirs = [src_root / split / 'labels', src_root / 'labels' / split, src_root / 'labels']
    image_dir = next((p for p in image_dirs if p.exists()), None)
    label_dir = next((p for p in label_dirs if p.exists()), None)
    if image_dir is None:
        return 0

    dst_img = dst_root / 'images' / split
    dst_lbl = dst_root / 'labels' / split
    dst_img.mkdir(parents=True, exist_ok=True)
    dst_lbl.mkdir(parents=True, exist_ok=True)

    count = 0
    for img in image_dir.glob('*'):
        if img.suffix.lower() not in {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}:
            continue
        prefix = src_root.name.replace('-', '_')
        new_img = dst_img / f'{prefix}_{img.name}'
        shutil.copy2(img, new_img)

        src_label = (label_dir / f'{img.stem}.txt') if label_dir else None
        dst_label = dst_lbl / f'{new_img.stem}.txt'
        if src_label and src_label.exists():
            lines = []
            for line in src_label.read_text(encoding='utf-8', errors='replace').splitlines():
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                try:
                    coords = [float(p) for p in parts[1:5]]
                except ValueError:
                    continue
                if all(0.0 <= c <= 1.0 for c in coords):
                    lines.append(' '.join(parts[:5]))
            dst_label.write_text('\n'.join(lines) + ('\n' if lines else ''), encoding='utf-8')
        else:
            dst_label.write_text('', encoding='utf-8')
        count += 1
    return count

def validate_labels(root: Path) -> dict:
    problems = []
    total = 0
    for txt in sorted(root.glob('labels/**/*.txt')):
        text = txt.read_text(encoding='utf-8', errors='replace')
        if not text.strip():
            continue
        for lineno, line in enumerate(text.splitlines(), start=1):
            parts = line.strip().split()
            if len(parts) < 5:
                problems.append((txt, lineno, 'too few fields'))
                continue
            try:
                coords = [float(p) for p in parts[1:5]]
            except ValueError:
                problems.append((txt, lineno, 'non-float coordinate'))
                continue
            if not all(0.0 <= c <= 1.0 for c in coords):
                problems.append((txt, lineno, 'coordinate outside [0,1]'))
            total += 1
    return {'total_labels': total, 'problems': problems}

if MERGED.exists():
    shutil.rmtree(MERGED)

uploaded_dirs = []
for path in sorted(KAGGLE_INPUT.glob('**/data.yaml')) + sorted(KAGGLE_INPUT.glob('**/dataset.yaml')):
    uploaded_dirs.append(path.parent)
uploaded_dirs = sorted(set(uploaded_dirs))

print('Dataset roots used for merge:')
for d in uploaded_dirs:
    print('-', d)

if not uploaded_dirs:
    raise RuntimeError('No YOLO dataset roots found under /kaggle/input. Upload a dataset with data.yaml or dataset.yaml first.')

merged_names = None
for dataset_dir in uploaded_dirs:
    data_yaml = find_dataset_yaml(dataset_dir)
    raw = yaml.safe_load(data_yaml.read_text(encoding='utf-8'))
    names = normalize_names(raw.get('names'))
    if merged_names is None:
        merged_names = names
    elif names != merged_names:
        print('WARNING: class names differ between datasets:', dataset_dir)
    print('Merging:', dataset_dir.name)
    for split in ['train', 'valid', 'test']:
        n = copy_split(dataset_dir, split, MERGED)
        print(f'  {split}: {n} images')

if merged_names is None:
    merged_names = {0: 'vehicle'}

val_split = 'valid' if (MERGED / 'images' / 'valid').exists() else 'train'
additional = {}
if (MERGED / 'images' / 'test').exists():
    additional['test'] = 'images/test'

data = {
    'path': str(MERGED),
    'train': 'images/train',
    'val': f'images/{val_split}',
    'names': merged_names,
}
if additional:
    data.update(additional)

(MERGED / 'data.yaml').write_text(yaml.safe_dump(data, sort_keys=False), encoding='utf-8')
print('')
print('Merged data.yaml:')
print((MERGED / 'data.yaml').read_text(encoding='utf-8'))

validation = validate_labels(MERGED)
print('')
print('Label validation: total label lines =', validation['total_labels'])
if validation['problems']:
    print('Found label problems:')
    for txt, lineno, reason in validation['problems'][:20]:
        print(f'  {txt.relative_to(MERGED)}:{lineno} {reason}')
    raise RuntimeError('Fix label problems before training.')
else:
    print('Label validation passed. Ready for training.')


In [ ]:
# Train the vehicle detector on Kaggle T4 using YOLOv26
import torch
from ultralytics import YOLO

def get_device() -> str:
    n = torch.cuda.device_count()
    if n >= 1:
        return '0'
    return 'cpu'

def load_yolo_with_fallback(primary: str, fallback: str):
    try:
        print('Trying base model:', primary)
        return YOLO(primary), primary
    except Exception as exc:
        print(f'Primary model {primary} failed: {exc}')
        print('Falling back to', fallback)
        return YOLO(fallback), fallback

DEVICE = get_device()
print('Training device:', DEVICE)

vehicle_model, vehicle_base = load_yolo_with_fallback('yolo26n.pt', 'yolo11n.pt')
print('Using base model:', vehicle_base)

SMOKE_TEST = True
if SMOKE_TEST:
    print('Running 1-epoch smoke test to validate dataset and labels')
    vehicle_model.train(
        data=str(MERGED / 'data.yaml'),
        epochs=1,
        imgsz=640,
        batch=16,
        device=DEVICE,
        project=str(RUNS),
        name='vehicle_yolov26n_smoke',
        save_period=1,
        exist_ok=True,
        workers=4,
        cache=True,
    )
    print('Smoke test complete. Inspect', RUNS / 'vehicle_yolov26n_smoke')

print('Starting full training using YOLOv26')
vehicle_model.train(
    data=str(MERGED / 'data.yaml'),
    epochs=80,
    imgsz=640,
    batch=16,
    accumulate=2,
    device=DEVICE,
    patience=20,
    project=str(RUNS),
    name='vehicle_yolov26n_v1',
    save_period=5,
    exist_ok=True,
    workers=4,
    cache=True,
)

best_vehicle = RUNS / 'vehicle_yolov26n_v1' / 'weights' / 'best.pt'
print('Final checkpoint path:', best_vehicle)


In [ ]:
# Evaluate the trained vehicle model
from ultralytics import YOLO

best_vehicle_path = RUNS / 'vehicle_yolov26n_v1' / 'weights' / 'best.pt'
if best_vehicle_path.exists():
    model = YOLO(str(best_vehicle_path))
    metrics = model.val(data=str(MERGED / 'data.yaml'), workers=4)
    print(f'Vehicle mAP50:     {metrics.box.map50:.3f}')
    print(f'Vehicle mAP50-95:  {metrics.box.map:.3f}')
    print(f'Vehicle precision: {metrics.box.mp:.3f}')
    print(f'Vehicle recall:    {metrics.box.mr:.3f}')
else:
    print('No trained best.pt found. Run the training cell first.')


In [ ]:
# Package the trained vehicle model and artifacts
import zipfile

best_vehicle_path = RUNS / 'vehicle_yolov26n_v1' / 'weights' / 'best.pt'
if best_vehicle_path.exists():
    shutil.copy2(best_vehicle_path, ARTIFACTS / 'vehicle_detector_best.pt')
    print('Copied vehicle model to', ARTIFACTS / 'vehicle_detector_best.pt')
else:
    print('WARNING: vehicle best.pt not found. Training may not have completed.')

manifest = {
    'vehicle_model_base': vehicle_base,
    'vehicle_model_path': str(best_vehicle_path) if best_vehicle_path.exists() else None,
    'merged_dataset': str(MERGED),
}
(ARTIFACTS / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')

zip_path = WORK / 'jsep_kaggle_vehicle_artifacts.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(ARTIFACTS.glob('*')):
        z.write(p, arcname=p.name)

print('Artifact zip created:', zip_path)
